# Module 10 — LLM Inference Lifecycle

Trace Qwen2.5-0.5B-Instruct from tokenization through prefill, logits, cached autoregressive decoding, and KV-cache accounting on one NVIDIA T4.

Install dependencies in the Lightning AI terminal if needed:

```bash
pip install -U 'transformers>=4.45,<5' accelerate safetensors
```

In [ ]:
# Complete standalone lifecycle implementation; no import from the companion .py file.
import statistics
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
assert torch.cuda.is_available(), 'Select a T4 GPU runtime.'
device = torch.device('cuda:0')

def load_model(device, model_id=MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device).eval()
    return tokenizer, model

def tokenize_prompt(tokenizer, prompt, device):
    messages = [{'role': 'user', 'content': prompt}]
    encoded = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
    return {key: value.to(device) for key, value in encoded.items()}

def benchmark_prefill(model, inputs, repeats=10):
    with torch.inference_mode():
        for _ in range(3): model(**inputs, use_cache=True, return_dict=True)
        torch.cuda.synchronize(device); samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True); start.record()
            model(**inputs, use_cache=True, return_dict=True); end.record(); end.synchronize(); samples.append(start.elapsed_time(end))
    return statistics.median(samples)

def generate_greedy(model, tokenizer, inputs, max_new_tokens=16):
    with torch.inference_mode():
        result = model(**inputs, use_cache=True, return_dict=True); cache = result.past_key_values
        next_token = result.logits[:, -1:].argmax(dim=-1); generated = [next_token]; mask = inputs['attention_mask']
        times = []
        for _ in range(max_new_tokens - 1):
            if next_token.item() == tokenizer.eos_token_id: break
            mask = torch.cat([mask, mask.new_ones((1, 1))], dim=-1)
            position = torch.tensor([cache.get_seq_length()], dtype=torch.long, device=next_token.device)
            torch.cuda.synchronize(device); start = time.perf_counter()
            result = model(input_ids=next_token, attention_mask=mask, past_key_values=cache, cache_position=position, use_cache=True, return_dict=True)
            torch.cuda.synchronize(device); times.append((time.perf_counter() - start) * 1000)
            cache = result.past_key_values; next_token = result.logits[:, -1:].argmax(dim=-1); generated.append(next_token)
    return tokenizer.decode(torch.cat(generated, dim=1)[0].tolist(), skip_special_tokens=True), cache, times

def cache_bytes(cache):
    total = 0; layers = cache.layers if hasattr(cache, 'layers') else cache
    for layer in layers:
        tensors = (layer.keys, layer.values) if hasattr(layer, 'keys') else layer
        for tensor in tensors: total += tensor.numel() * tensor.element_size()
    return total

print('GPU:', torch.cuda.get_device_name(device))
print('CUDA capability:', torch.cuda.get_device_capability(device))
print('PyTorch:', torch.__version__)

In [ ]:
tokenizer, model = load_model(device, MODEL_ID)
prompt = 'What is the capital of France? Answer briefly.'
inputs = tokenize_prompt(tokenizer, prompt, device)
print('Input IDs shape:', tuple(inputs['input_ids'].shape))
print('Input IDs:', inputs['input_ids'][0].tolist())
print('Embedding shape:', tuple(model.get_input_embeddings()(inputs['input_ids']).shape))

In [ ]:
with torch.inference_mode():
    outputs = model(**inputs, use_cache=True, return_dict=True)
next_logits = outputs.logits[:, -1, :]
next_token = next_logits.argmax(dim=-1, keepdim=True)
print('Logits shape:', tuple(outputs.logits.shape))
print('Next-token logits:', tuple(next_logits.shape))
print('First selected token:', repr(tokenizer.decode(next_token[0].tolist())))
print('Cache sequence length:', outputs.past_key_values.get_seq_length())

In [ ]:
prefill_ms = benchmark_prefill(model, inputs, repeats=10)
prompt_tokens = inputs['input_ids'].shape[1]
print('Median prefill ms:', round(prefill_ms, 3))
print('Prompt tokens/sec:', round(prompt_tokens / (prefill_ms / 1000), 1))

response, cache, decode_times_ms = generate_greedy(
    model, tokenizer, inputs, max_new_tokens=16
)
print('Generated response:', response)
print('Cache sequence length:', cache.get_seq_length())
print('Logical KV-cache MiB:', round(cache_bytes(cache) / 2**20, 4))
if decode_times_ms:
    print('Mean incremental decode ms:', sum(decode_times_ms) / len(decode_times_ms))

## What to record

Record the model load time, actual token IDs, embedding/logit shapes, median prefill time, generated text, cache sequence length, logical KV-cache bytes, and mean incremental decode time. Compare the first forward pass with later one-token decode passes.